***
# Homework 8: HTML and JSON

*Course:** STAT 606 - Computing in Data Science and Statistics SP24

**Name:** Shrivats Sudhir

**NetID:** ssudhir2

**Email:** ssudhir2@wisc.edu

**Collaborators:** Samuel Merten, Amy Merkelz

**Date:** March 31st, 2024
***

In [1]:
from urllib.error import HTTPError
import urllib.request
from bs4 import BeautifulSoup

import pandas as pd

## 1.) Warmup: Parsing HTML (spent $\approx$ 10 minutes)

**Let’s get started using `BeautifulSoup` with a couple of simple exercises. Both of the following subproblems ask you to retrieve the HTML from a URL given by an argument `s` and determine some simple information about that HTML.** 

**In both functions, you should raise an appropriate error in the event that `s` is not a string, and you should raise an `HTTPError` with an appropriate error message in the event that the success code that
results from trying to access the URL is not code 200.** 

**You may rely on `requests` and/or `urllib` to raise an error for you in the event that `s` is a string but does not encode a valid URL.**

**Write a function `get_page_title` that takes a string `s` as its only argument and returns a string.** 

**Your function should try to treat `s` as a URL, and return the string stored in the title tag of the HTML page stored at that URL.**

**In the unlikely event that the URL has more than one title tag, your function should return the text stored in the first one. If no such title exists, your function should return `None`.**

**A good way to test this function is to simply check that your function returns the string matching the title in your browser– the page title will always be displayed in the tab in which you have the page open.**

In [2]:
def get_page_title(s):

    if not isinstance(s, (str, )):
        raise TypeError(f'URL ({s}) must be a string.')
    
    try:
        response = urllib.request.urlopen(s)
        if response.getcode() != 200:
            raise HTTPError(f'Could not open URL ({s}).')
    except HTTPError as error:
        raise HTTPError(f'HTTP error occurred: {error}')
    
    html = response.read()
    soup = BeautifulSoup(html, 'html.parser')

    return soup.title.string

**Write a function `count_links` that takes a string `s` encoding a URL as its only argument and returns an integer corresponding to the number of hyperlinks on the webpage stored at that URL.** 

**The homework instructions page linked to above is an easy page to test your code on– there are only three links.**

In [3]:
def count_links(s):
    
    if not isinstance(s, (str, )):
        raise TypeError(f'URL ({s}) must be a string.')
    
    try:
        response = urllib.request.urlopen(s)
        if response.getcode() != 200:
            raise HTTPError(f'Could not open URL ({s}).')
    except HTTPError as error:
        raise HTTPError(f'HTTP error occurred: {error}')
    
    html = response.read()
    soup = BeautifulSoup(html, 'html.parser')

    return len(soup.find_all('a'))

## 2.) Retrieving Data from the Web (7 points, spent $\approx$ )

**In this problem, we’ll scrape data from Wikipedia using `BeautifulSoup`. Documentation for `BeauitfulSoup` can be found at *https://www.crummy.com/software/BeautifulSoup/bs4/doc/*.** 

**As mentioned in lecture, there is another package, called requests, which is becoming quite popular, which you are welcome to use for this problem instead, if you wish. Documentation for the requests package can be found at *http://docs.python-requests.org/en/master/*.**

**Suppose you are trying to choose a city to vacation in. A major factor in your decision is weather. Conveniently, lots of weather information is present in the Wikipedia articles for most world cities.** 

**Your job in this problem is to use `BeautifulSoup` to retrieve weather information from Wikipedia articles. We should note that in practice, such information is more easily obtained from, for example, the National Oceanic and Atmospheric Administration (NOAA) in the case of American cities, and from analogous organizations in other countries.**

**Look at a few Wikipedia pages corresponding to cities. For example:**

* *https://en.wikipedia.org/wiki/Madison,_Wisconsin*

* *https://en.wikipedia.org/wiki/Buenos_Aires*

* *https://en.wikipedia.org/wiki/Harbin*

**Note that most city pages include a table titled something like “Climate data for [Cityname] (normals YYYY-YYYY, extremes YYYY-YYYY)” Find a Wikipedia page for a city that includes such a table (such as one of the three above).**

**In your jupyter notebook, open the URL and read the HTML using either `urllib` or `requests`, and parse it with `BeautifulSoup` using the standard parser, `html.parser`.**

**Have a look at the parsed HTML and find the climate data table, which will have the tag table and will contain a child tag `th` containing a string similar to**

`Climate data for [Cityname] (normals YYYY-YYYY, extremes YYYY-YYYY).`

**Find the node in the `BeautifulSoup` object corresponding to this table. What is the structure of this node of the tree (e.g., how many children does the table have, what are their tags, etc.)? You may want to learn a bit about the structure of HTML tables by looking at the resources available on these websites:**

* *https://developer.mozilla.org/en-US/docs/Web/HTML/Element/table*

* *https://www.w3schools.com/html/html_tables.asp*

* *https://www.w3.org/TR/html401/struct/tables.html*

After inspecting element for url = 'https://en.wikipedia.org/wiki/Madison,_Wisconsin', I found the table for `Climate data for {City Name}` and pasted the CSS Selector and XPATH:

CSS Selector: $\texttt{.mw-content-ltr > div:nth-child(110)}$

XPATH: $\texttt{/html/body/div[2]/div/div[3]/main/div[3]/div[3]/div[1]/div[16]}$

I also noticed the following:

* It is enclosed between `<div> <table> <tbody> ... </tbody> </table> </div>`

* The header is enclosed between `<tbody> <tr> ... </tr> </tbody>` and contains `<th colspan="14">`.

* `colspan="14"` always contains the following columns, (1.) Months, (2.) - (13.) Jan - Dec, (14.) Year.

**Write a function `retrieve_climate_table` that takes as its only argument a string representing a URL, and returns the `BeautifulSoup` tag object corresponding to the climate data table (if it exists in the page) and returns `None` if no such table exists on the page.**

**You should check that the URL is retrieved successfully, and raise an error if `urllib2` fails to successfully read the website.** 

**You may notice that some city pages include more than one climate data table or several nested tables (see, for example, https://en.wikipedia.org/wiki/Los_Angeles). In this case, your function may arbitrarily choose one of the tables to return as a BeautifulSoup object.**

In [4]:
def retrieve_climate_table(s):

    if not isinstance(s, (str, )):
        raise TypeError(f'URL ({s}) must be a string.')
    
    try:
        response = urllib.request.urlopen(s)
    except HTTPError as error:
        raise HTTPError(f'HTTP error occurred: {error}')
    
    response = urllib.request.urlopen(s)   
    html = response.read()
    soup = BeautifulSoup(html, 'html.parser')

    for table in soup.find_all('table'):
        th = table.find_all('th')
        for i in th:
            if 'colspan' in i.attrs.keys() and '14' in i.attrs.values():
                return table
    return None    

**As you look at some of the climate data tables, you may notice that different cities' tables contain different information. For example, not all cities include snowfall data.** 

**Write a function `list_climate_table_row_names` that takes as its only argument a Wikipedia URL and returns a list of the row names of the climate data table, or returns `None` if no such table exists. The list returned by your function should, ideally, consist solely of Python strings (either Unicode or ASCII), and should not include any BeautifulSoup objects or HTML, and the strings should not have any trailing whitespace (Hint: see the `BeautifulSoup` method get_text()).** 

**The list returned by your script should not include an entry corresponding to the `Climate data for...` row in the table.** 

**Second hint: you are looking for HTML table header (`th`) objects. The HTML attribute `scope` is your friend here, because in the context of an HTML table it tells you when a `th` tag is the header of a row or a column.**

In [5]:
def list_climate_table_row_names(s):
    row_names = []
    table_html = retrieve_climate_table(s)
    for row in table_html.find_all('th'):
        if 'scope' in row.attrs.keys() and 'row' in row.attrs.values():
            if row.text.strip() != 'Month':
                row_names.append(row.text.strip())
            else:
                next
    return row_names

**The next natural step would be to write a function that takes a URL and a row name and retrieves the data from that row of the climate data table (if the table exists and has that row name). Doing this would require some complicated string wrangling to get right, so I’ll spare you the trouble. Instead, please briefly describe either in pseudo code or in plain English how you would accomplish this, using the two functions you wrote above and the tools available to you in the `BeautifulSoup` package.** 

**Note: just to be clear, you do not have to write any code for this last step. Of course, if you want a challenge, you are welcome to try writing this code, but it is not required for this assignment**

I have abstracted the function described above so that the only input required is the URL. My function does the following:

* Gathers all the row names by running the `list_climate_table_row_names()` written above.

* Gathers all column elements (for each row name) inside a list called `elements`. Note that, I found that after inspecting element, each cell value is of the form `<td> number <br> (number) </td>`.

* Finally, as I know that each splices of 13 numbers are the row data for each row name, I initialized an empty dictionary, appended row names and its element values, and return the pd.DataFrame associated to the dictionary.

In [6]:
def list_column_data_rows(s):

    row_names = list_climate_table_row_names(s)
    
    elements = []
    table_html = retrieve_climate_table(s)   
    for row in table_html.find_all('th'):
        for col in table_html.find_all('td'):
            if row.text.strip() in row_names:
                elements.append(col.text.strip())

    data = dict({'Month':['January', 
                          'February',
                          'March', 
                          'April', 
                          'May', 
                          'June', 
                          'July', 
                          'August', 
                          'September', 
                          'October', 
                          'November', 
                          'December',
                          'Year']})

    for i in range(len(row_names)):
        data[row_names[i]] = elements[i * 13 : (i+1) * 13]
    
    return pd.DataFrame(data)

In [7]:
s = 'https://en.wikipedia.org/wiki/Madison,_Wisconsin'
list_column_data_rows(s)

,Month,Record high °F (°C),Mean maximum °F (°C),Mean daily maximum °F (°C),Daily mean °F (°C),Mean daily minimum °F (°C),Mean minimum °F (°C),Record low °F (°C),Average precipitation inches (mm),Average snowfall inches (cm),Average precipitation days (≥ 0.01 in),Average snowy days (≥ 0.1 in),Average relative humidity (%),Mean monthly sunshine hours,Percent possible sunshine
0,January,58(14),46.2(7.9),27.0(−2.8),19.4(−7.0),11.8(−11.2),−10.6(−23.7),−37(−38),1.47(37),13.7(35),10.6,10.1,74.5,143.0,49
1,February,70(21),51.3(10.7),31.2(−0.4),23.0(−5.0),14.9(−9.5),−5.5(−20.8),−29(−34),1.52(39),12.8(33),9.7,8.6,73.1,152.3,52
2,March,83(28),67.1(19.5),43.6(6.4),34.4(1.3),25.1(−3.8),4.2(−15.4),−29(−34),2.26(57),7.0(18),10.6,5.3,71.4,187.3,51
3,April,94(34),79.1(26.2),56.9(13.8),46.3(7.9),35.8(2.1),21.3(−5.9),0(−18),3.78(96),2.6(6.6),12.6,1.9,66.3,206.7,51
4,May,101(38),85.6(29.8),69.0(20.6),58.1(14.5),47.1(8.4),32.1(0.1),19(−7),4.10(104),0.1(0.25),12.7,0.1,65.8,263.1,58
5,June,101(38),91.0(32.8),78.6(25.9),68.0(20.0),57.4(14.1),43.2(6.2),31(−1),5.28(134),0.0(0.0),11.7,0.0,68.3,293.1,64
6,July,107(42),92.2(33.4),82.1(27.8),71.9(22.2),61.6(16.4),49.9(9.9),36(2),4.51(115),0.0(0.0),10.2,0.0,71.0,304.9,66
7,August,102(39),90.4(32.4),79.9(26.6),69.7(20.9),59.5(15.3),48.1(8.9),35(2),4.16(106),0.0(0.0),9.4,0.0,74.4,270.2,63
8,September,99(37),87.6(30.9),72.9(22.7),62.0(16.7),51.0(10.6),35.8(2.1),25(−4),3.43(87),0.0(0.0),9.2,0.0,76.8,213.8,57
9,October,90(32),79.4(26.3),59.6(15.3),49.7(9.8),39.8(4.3),25.3(−3.7),12(−11),2.77(70),0.6(1.5),10.1,0.5,73.2,172.5,50
